# InfluencerRank v3 FIXED: No Temporal Leakage + Optimized Training

**CRITICAL FIXES:**
1. **PROGRESSIVE TEMPORAL FEATURES** - Each month uses only PAST data (requires v3 graphs)
2. **CORRECT LEAKY INDICES** - All 16 leaky features properly identified
3. **GPU MEMORY OPTIMIZATION** - Pre-load graphs, cache embeddings per epoch
4. **EFFICIENT TRAINING** - Reduced redundant computation

**Previous v3 Bug:** Graphs had temporal features from months 0-8 applied to ALL months,
causing Jan-Sep graphs to contain future information. This inflated NDCG to 0.71+ at epoch 1.

**Expected Performance (Honest):**
- With leakage fix: 0.55-0.62 NDCG@50 (down from inflated 0.71)
- Paper target: 0.720 NDCG@50 (requires better architecture or more data)

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ndcg_score
import warnings
import time
warnings.filterwarnings('ignore')

from torch_geometric.nn import GCNConv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configuration (FIXED LEAKY INDICES)

In [ ]:
# Data paths - USE V3 GRAPHS WITH PROGRESSIVE TEMPORAL FEATURES!
GRAPH_DIR = '/kaggle/input/graphsv3/graphs_enhanced_v3'  # MUST use v3 graphs!

# CRITICAL: CORRECTED leaky feature indices
# Feature layout (37 total):
# [0-2]   Static: log_followers, log_followees, follower_ratio
# [3-10]  Category: one-hot (8 dims)
# [11-24] Temporal: 14 dims (ALL contain past engagement/likes info!)
# [25-36] Monthly: 12 dims

LEAKY_INDICES = [
    # ALL temporal features (they encode engagement patterns)
    11,  # engagement_trend
    12,  # likes_trend
    13,  # engagement_variance
    14,  # likes_variance
    15,  # engagement_consistency
    16,  # likes_consistency
    17,  # engagement_momentum
    18,  # likes_momentum
    19,  # engagement_peak
    20,  # likes_peak
    21,  # activity_rate (WAS MISSING in v3!)
    22,  # posting_consistency (WAS MISSING in v3!)
    23,  # engagement_growth
    24,  # likes_growth
    # Monthly features that leak target
    26,  # log_avg_likes (CORRECT index!)
    27,  # log_avg_comments (WAS MISSING - wrong index in v3!)
]

print(f"CORRECTED LEAKY FEATURE INDICES: {LEAKY_INDICES}")
print(f"Total: {len(LEAKY_INDICES)} features (16 total, vs 14 in buggy v3)")
print("\nNEW additions (were missing in v3):")
print("  - 21: activity_rate (% months active)")
print("  - 22: posting_consistency (variance in posts)")
print("  - 27: log_avg_comments (correlates with engagement)")
print("\nFIXED:")
print("  - v3 claimed 25=log_avg_likes, but 25=num_posts!")
print("  - v3 claimed 26=log_avg_comments, but 26=log_avg_likes!")
print("  - Correct: 26=log_avg_likes, 27=log_avg_comments")

# Architecture
INPUT_DIM = 37
GNN_HIDDEN = 128
GNN_OUT = 128
RNN_HIDDEN = 128
DROPOUT = 0.5

# Training (optimized)
BATCH_SIZE = 64  # Increased from 32 (more efficient with caching)
LIST_SIZE = 10
LEARNING_RATE = 0.001
NUM_EPOCHS = 200
EARLY_STOP_PATIENCE = 30
WEIGHT_DECAY = 1e-5
CACHE_EMBEDDINGS_PER_EPOCH = True  # NEW: Cache GCN embeddings

# Temporal settings
TRAINING_MONTHS = 9
TARGET_MONTH = 9

# Ensemble
ENSEMBLE_SEEDS = [42, 123, 456, 789, 2024]

# Data split
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

print(f"\nArchitecture: GCN({INPUT_DIM}->{GNN_HIDDEN}->{GNN_OUT}) -> GRU -> Attention -> FC")
print(f"Training: LR={LEARNING_RATE}, Batch={BATCH_SIZE}, ListSize={LIST_SIZE}")
print(f"\n** V3 FIXED: No temporal leakage, correct indices, optimized training **")

## 3. Load Graphs (V3 Required!)

In [ ]:
month_names = ["jan", "feb", "mar", "apr", "may", "jun", 
               "jul", "aug", "sep", "oct", "nov", "dec"]

print("Loading V3 graphs (with progressive temporal features)...")
all_graphs_data = []

for month_idx, month in enumerate(month_names):
    path = os.path.join(GRAPH_DIR, f"{month}_graph.pt")
    data = torch.load(path, weights_only=False)
    all_graphs_data.append(data)
    
    # VERIFY progressive temporal features
    temporal_months = data['metadata'].get('temporal_months_used', 'UNKNOWN')
    no_leak = data['metadata'].get('no_future_leakage', False)
    
    print(f"  {month.upper()}: {data['graph']['influencer'].x.shape[0]} influencers, "
          f"temporal from {temporal_months} past months, no_leak={no_leak}")

# CRITICAL VERIFICATION
if not all_graphs_data[0]['metadata'].get('no_future_leakage', False):
    print("\n" + "="*60)
    print("WARNING: Graphs may not have progressive temporal features!")
    print("Please regenerate using build_enhanced_graphs_v3_fixed.py")
    print("="*60)

print(f"\nLoaded {len(all_graphs_data)} monthly graphs")

In [ ]:
# Zero out leaky features
print("\n" + "="*60)
print("REMOVING DATA LEAKAGE (CORRECTED INDICES)")
print("="*60)

print(f"Zeroing out {len(LEAKY_INDICES)} leaky features at indices: {LEAKY_INDICES}")

# Show before
print(f"\nBefore zeroing (Oct graph, first influencer):")
oct_features = all_graphs_data[9]['graph']['influencer'].x
print(f"  Feature 11 (engagement_trend): {oct_features[0, 11]:.6f}")
print(f"  Feature 21 (activity_rate): {oct_features[0, 21]:.6f}")
print(f"  Feature 26 (log_avg_likes): {oct_features[0, 26]:.6f}")
print(f"  Feature 27 (log_avg_comments): {oct_features[0, 27]:.6f}")

# Zero out leaky features in ALL months
for month_idx, data_package in enumerate(all_graphs_data):
    features = data_package['graph']['influencer'].x
    features[:, LEAKY_INDICES] = 0.0

# Show after
print(f"\nAfter zeroing (Oct graph, first influencer):")
oct_features = all_graphs_data[9]['graph']['influencer'].x
print(f"  Feature 11 (engagement_trend): {oct_features[0, 11]:.6f}")
print(f"  Feature 21 (activity_rate): {oct_features[0, 21]:.6f}")
print(f"  Feature 26 (log_avg_likes): {oct_features[0, 26]:.6f}")
print(f"  Feature 27 (log_avg_comments): {oct_features[0, 27]:.6f}")

# Verify ALL months
print(f"\nVerifying leakage removal in ALL months:")
for month_idx in range(TRAINING_MONTHS):
    features = all_graphs_data[month_idx]['graph']['influencer'].x
    leaky_sum = features[:, LEAKY_INDICES].abs().sum().item()
    assert leaky_sum == 0, f"Month {month_idx} still has leaky features!"
    print(f"  Month {month_idx}: OK (leaky features sum = 0)")

print(f"\n✅ LEAKAGE REMOVED! {len(LEAKY_INDICES)} features zeroed in all months.")
print(f"Effective features: {INPUT_DIM - len(LEAKY_INDICES)} non-zero dimensions")
print("="*60)

## 4. Data Split

In [ ]:
target_data = all_graphs_data[TARGET_MONTH]
all_influencers = list(target_data['maps']['influencer'].keys())

print(f"Total influencers in target month (October): {len(all_influencers)}")

np.random.seed(42)
np.random.shuffle(all_influencers)

n = len(all_influencers)
n_train = int(TRAIN_RATIO * n)
n_val = int(VAL_RATIO * n)

train_influencers = set(all_influencers[:n_train])
val_influencers = set(all_influencers[n_train:n_train + n_val])
test_influencers = set(all_influencers[n_train + n_val:])

assert len(train_influencers & val_influencers) == 0
assert len(train_influencers & test_influencers) == 0
assert len(val_influencers & test_influencers) == 0

print(f"\nData Split:")
print(f"  Train: {len(train_influencers)} ({100*len(train_influencers)/n:.1f}%)")
print(f"  Val:   {len(val_influencers)} ({100*len(val_influencers)/n:.1f}%)")
print(f"  Test:  {len(test_influencers)} ({100*len(test_influencers)/n:.1f}%)")

train_influencers_list = list(train_influencers)
val_influencers_list = list(val_influencers)
test_influencers_list = list(test_influencers)

## 5. Train-Only Normalization

In [ ]:
print("Performing train-only normalization...")

train_features = []
for month_idx in range(TRAINING_MONTHS):
    month_data = all_graphs_data[month_idx]
    features = month_data['graph']['influencer'].x
    influencer_map = month_data['maps']['influencer']
    
    for inf_name in train_influencers:
        if inf_name in influencer_map:
            local_idx = influencer_map[inf_name]
            train_features.append(features[local_idx].numpy())

train_features = np.vstack(train_features)
print(f"  Collected {train_features.shape[0]} training feature vectors")

scaler = StandardScaler()
scaler.fit(train_features)

print(f"  Scaler fitted on training data only")

In [ ]:
# Apply normalization to ONLY influencer features
print("Applying normalization to INFLUENCER features ONLY...")

for month_idx, data_package in enumerate(all_graphs_data):
    features = data_package['graph']['influencer'].x
    normalized = scaler.transform(features.numpy())
    data_package['graph']['influencer'].x = torch.FloatTensor(normalized)

oct_features = all_graphs_data[9]['graph']['influencer'].x
print(f"  After normalization: Mean={oct_features.mean():.6f}, Std={oct_features.std():.6f}")

## 6. Graph Conversion (Optimized)

In [ ]:
def convert_hetero_to_homogeneous(graph, influencer_map):
    """Convert heterogeneous graph to homogeneous for GCN."""
    x_inf = graph['influencer'].x
    x_hash = graph['hashtag'].x if 'hashtag' in graph.node_types else torch.zeros((0, x_inf.shape[1]))
    x_user = graph['user'].x if 'user' in graph.node_types else torch.zeros((0, x_inf.shape[1]))
    x_obj = graph['object'].x if 'object' in graph.node_types else torch.zeros((0, x_inf.shape[1]))
    
    x = torch.cat([x_inf, x_hash, x_user, x_obj], dim=0)
    
    offset_inf = 0
    offset_hash = x_inf.shape[0]
    offset_user = offset_hash + x_hash.shape[0]
    offset_obj = offset_user + x_user.shape[0]
    
    offsets = {
        'influencer': offset_inf,
        'hashtag': offset_hash,
        'user': offset_user,
        'object': offset_obj
    }
    
    edge_list = []
    for edge_type in graph.edge_types:
        src_type, _, dst_type = edge_type
        if edge_type in graph.edge_index_dict:
            edges = graph[edge_type].edge_index.clone()
            edges[0] += offsets[src_type]
            edges[1] += offsets[dst_type]
            edge_list.append(edges)
            reverse_edges = torch.stack([edges[1], edges[0]], dim=0)
            edge_list.append(reverse_edges)
    
    if edge_list:
        edge_index = torch.cat(edge_list, dim=1)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    
    # Add self-loops
    num_nodes = x.shape[0]
    self_loops = torch.stack([torch.arange(num_nodes), torch.arange(num_nodes)])
    edge_index = torch.cat([edge_index, self_loops], dim=1)
    
    influencer_global_indices = {}
    for name, local_idx in influencer_map.items():
        influencer_global_indices[name] = local_idx + offset_inf
    
    return x, edge_index, influencer_global_indices

print("Graph conversion function defined.")

In [ ]:
# Pre-convert graphs and PRE-LOAD TO GPU (optimization)
print("\nPre-converting graphs and loading to GPU...")

converted_graphs = []

for month_idx in range(TRAINING_MONTHS):
    data_package = all_graphs_data[month_idx]
    graph = data_package['graph']
    influencer_map = data_package['maps']['influencer']
    
    x, edge_index, global_indices = convert_hetero_to_homogeneous(graph, influencer_map)
    
    # PRE-LOAD TO GPU (optimization - avoids transfer every batch)
    converted_graphs.append({
        'x': x.to(device),  # Pre-loaded!
        'edge_index': edge_index.to(device),  # Pre-loaded!
        'global_indices': global_indices
    })
    
    print(f"  Month {month_idx}: {x.shape[0]} nodes, {edge_index.shape[1]} edges (on GPU)")

print(f"\n✅ All {len(converted_graphs)} graphs pre-loaded to GPU")

## 7. Model Architecture

In [ ]:
class SimpleGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x


class SimpleAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention_fc = nn.Linear(hidden_size, 1)
    
    def forward(self, hidden_states, lengths):
        batch_size, seq_len, _ = hidden_states.shape
        scores = self.attention_fc(hidden_states).squeeze(-1)
        mask = torch.arange(seq_len, device=hidden_states.device).expand(batch_size, -1)
        mask = mask < lengths.unsqueeze(1)
        scores = scores.masked_fill(~mask, -1e9)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)
        return context


class InfluencerRankModelV3Fixed(nn.Module):
    """V3 Fixed: End-to-end with optimized caching."""
    
    def __init__(self, input_dim, gnn_hidden, gnn_out, rnn_hidden, dropout=0.5):
        super().__init__()
        self.gcn = SimpleGCN(input_dim, gnn_hidden, gnn_out, dropout)
        self.rnn = nn.GRU(
            input_size=gnn_out,
            hidden_size=rnn_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
            dropout=0.0
        )
        self.attention = SimpleAttention(rnn_hidden)
        self.fc1 = nn.Linear(rnn_hidden, rnn_hidden // 2)
        self.fc2 = nn.Linear(rnn_hidden // 2, 1)
        self.dropout = nn.Dropout(dropout)
    
    def encode_graphs(self, converted_graphs):
        """Encode ALL graphs at once (cache per epoch)."""
        embeddings = []
        for cg in converted_graphs:
            emb = self.gcn(cg['x'], cg['edge_index'])
            embeddings.append(emb)
        return embeddings
        
    def forward_temporal(self, sequences, lengths):
        packed = pack_padded_sequence(sequences, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.rnn(packed)
        rnn_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        context = self.attention(rnn_output, lengths.to(rnn_output.device))
        x = F.relu(self.fc1(context))
        x = self.dropout(x)
        score = self.fc2(x).squeeze(-1)
        return score


print("Model architecture defined (with caching optimization).")

## 8. Loss and Metrics

In [ ]:
def listwise_ranking_loss(y_pred, y_true):
    pred_diff = y_pred.unsqueeze(1) - y_pred.unsqueeze(0)
    true_diff = y_true.unsqueeze(1) - y_true.unsqueeze(0)
    mask = (true_diff > 0).float()
    loss = -F.logsigmoid(pred_diff) * mask
    return loss.sum() / mask.sum().clamp(min=1)


def compute_ndcg_at_k(y_true, y_pred, k=50):
    if len(y_true) < 2:
        return 0.0
    y_true = np.array(y_true).reshape(1, -1)
    y_pred = np.array(y_pred).reshape(1, -1)
    k = min(k, len(y_true[0]))
    return ndcg_score(y_true, y_pred, k=k)


print("Loss and metrics defined.")

## 9. Training Utilities

In [ ]:
def get_ground_truth(influencer_names, target_data):
    ground_truth = []
    influencer_map = target_data['maps']['influencer']
    engagement_rates = target_data['ground_truth']['engagement_rate']
    
    for name in influencer_names:
        if name in influencer_map:
            local_idx = influencer_map[name]
            ground_truth.append(engagement_rates[local_idx].item())
        else:
            ground_truth.append(0.0)
    
    return torch.FloatTensor(ground_truth)


def sample_influencers(influencer_list, size):
    if len(influencer_list) >= size:
        indices = np.random.choice(len(influencer_list), size, replace=False)
        return [influencer_list[i] for i in indices]
    else:
        return influencer_list


print("Training utilities defined.")

## 10. Optimized Training Loop

In [ ]:
def train_single_model_optimized(seed, converted_graphs, train_list, val_list, test_list, all_graphs_data):
    """
    Optimized training with:
    1. Pre-loaded graphs on GPU
    2. Cached GCN embeddings per epoch (not per batch)
    3. Reduced memory transfers
    """
    print(f"\n{'='*60}")
    print(f"Training model with seed {seed}")
    print(f"{'='*60}")
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    model = InfluencerRankModelV3Fixed(INPUT_DIM, GNN_HIDDEN, GNN_OUT, RNN_HIDDEN, DROPOUT).to(device)
    
    print(f"Model: {sum(p.numel() for p in model.parameters())} parameters")
    print(f"  GCN: {sum(p.numel() for p in model.gcn.parameters())}")
    print(f"  RNN: {sum(p.numel() for p in model.rnn.parameters())}")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    
    best_val_ndcg = 0.0
    best_model_state = None
    patience_counter = 0
    target_data = all_graphs_data[TARGET_MONTH]
    
    print(f"\nStarting OPTIMIZED training (max {NUM_EPOCHS} epochs)...")
    print(f"  Caching GCN embeddings per epoch (not per batch)")
    
    epoch_times = []
    
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        model.train()
        epoch_losses = []
        
        # OPTIMIZATION: Cache GCN embeddings ONCE per epoch
        # This is the key speedup - compute graph embeddings once, reuse all batches
        month_embeddings = model.encode_graphs(converted_graphs)
        
        num_batches = max(1, len(train_list) // (BATCH_SIZE * LIST_SIZE))
        
        for batch_num in range(num_batches):
            optimizer.zero_grad()
            batch_loss = 0.0
            valid_samples = 0
            
            # Process BATCH_SIZE lists using CACHED embeddings
            for _ in range(BATCH_SIZE):
                batch_names = sample_influencers(train_list, LIST_SIZE)
                
                sequences = []
                valid_names = []
                
                for name in batch_names:
                    seq = []
                    for month_idx in range(TRAINING_MONTHS):
                        if name in converted_graphs[month_idx]['global_indices']:
                            global_idx = converted_graphs[month_idx]['global_indices'][name]
                            seq.append(month_embeddings[month_idx][global_idx])
                    
                    if len(seq) > 0:
                        sequences.append(torch.stack(seq))
                        valid_names.append(name)
                
                if len(sequences) < 2:
                    continue
                
                lengths = torch.LongTensor([s.shape[0] for s in sequences])
                padded = pad_sequence(sequences, batch_first=True)
                y_true = get_ground_truth(valid_names, target_data).to(device)
                y_pred = model.forward_temporal(padded, lengths)
                
                loss = listwise_ranking_loss(y_pred, y_true)
                batch_loss += loss
                valid_samples += 1
            
            if valid_samples > 0:
                (batch_loss / valid_samples).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_losses.append((batch_loss / valid_samples).item())
        
        # Validation (using same cached embeddings)
        avg_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        
        model.eval()
        with torch.no_grad():
            val_month_embeddings = model.encode_graphs(converted_graphs)
            
            val_sequences = []
            val_valid_names = []
            
            for name in val_list:
                seq = []
                for month_idx in range(TRAINING_MONTHS):
                    if name in converted_graphs[month_idx]['global_indices']:
                        global_idx = converted_graphs[month_idx]['global_indices'][name]
                        seq.append(val_month_embeddings[month_idx][global_idx])
                
                if len(seq) > 0:
                    val_sequences.append(torch.stack(seq))
                    val_valid_names.append(name)
            
            if len(val_sequences) >= 2:
                val_lengths = torch.LongTensor([s.shape[0] for s in val_sequences])
                val_padded = pad_sequence(val_sequences, batch_first=True)
                val_pred = model.forward_temporal(val_padded, val_lengths).cpu().numpy()
                val_true = get_ground_truth(val_valid_names, target_data).numpy()
                val_ndcg = compute_ndcg_at_k(val_true, val_pred, k=50)
            else:
                val_ndcg = 0.0
        
        scheduler.step(val_ndcg)
        
        if val_ndcg > best_val_ndcg:
            best_val_ndcg = val_ndcg
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        
        if (epoch + 1) % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            avg_epoch_time = np.mean(epoch_times[-10:])
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {avg_loss:.4f}, Val NDCG@50: {val_ndcg:.4f} "
                  f"(Best: {best_val_ndcg:.4f}), LR: {current_lr:.6f}, Time: {avg_epoch_time:.2f}s")
        
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # Load best model and evaluate on test
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        model = model.to(device)
    
    model.eval()
    with torch.no_grad():
        test_month_embeddings = model.encode_graphs(converted_graphs)
        
        test_sequences = []
        test_valid_names = []
        
        for name in test_list:
            seq = []
            for month_idx in range(TRAINING_MONTHS):
                if name in converted_graphs[month_idx]['global_indices']:
                    global_idx = converted_graphs[month_idx]['global_indices'][name]
                    seq.append(test_month_embeddings[month_idx][global_idx])
            
            if len(seq) > 0:
                test_sequences.append(torch.stack(seq))
                test_valid_names.append(name)
        
        if len(test_sequences) >= 2:
            test_lengths = torch.LongTensor([s.shape[0] for s in test_sequences])
            test_padded = pad_sequence(test_sequences, batch_first=True)
            test_pred = model.forward_temporal(test_padded, test_lengths).cpu().numpy()
            test_true = get_ground_truth(test_valid_names, target_data).numpy()
            test_ndcg = compute_ndcg_at_k(test_true, test_pred, k=50)
        else:
            test_ndcg = 0.0
            test_pred = np.array([])
            test_true = np.array([])
    
    print(f"\n  Final Results (Seed {seed}):")
    print(f"    Best Val NDCG@50: {best_val_ndcg:.4f}")
    print(f"    Test NDCG@50: {test_ndcg:.4f}")
    print(f"    Avg epoch time: {np.mean(epoch_times):.2f}s")
    
    return {
        'seed': seed,
        'model': model,
        'best_val_ndcg': best_val_ndcg,
        'test_ndcg': test_ndcg,
        'test_pred': test_pred,
        'test_true': test_true,
        'test_names': test_valid_names,
        'avg_epoch_time': np.mean(epoch_times)
    }


print("Optimized training loop defined.")
print("  - GCN embeddings cached once per epoch (not per batch)")
print("  - Graphs pre-loaded on GPU (no transfers during training)")
print("  - Expected speedup: 5-10x")

## 11. Train Ensemble

In [ ]:
print("\n" + "="*60)
print("STARTING V3 FIXED ENSEMBLE TRAINING")
print("="*60)
print(f"Training {len(ENSEMBLE_SEEDS)} models with seeds: {ENSEMBLE_SEEDS}")
print(f"\nKEY FIXES:")
print(f"  1. Progressive temporal features (no future leakage)")
print(f"  2. Correct leaky indices ({len(LEAKY_INDICES)} features zeroed)")
print(f"  3. Optimized training (cached embeddings, pre-loaded GPU)")
print(f"\nEXPECTED HONEST NDCG@50: 0.55-0.62 (down from inflated 0.71+)")

ensemble_results = []

for seed in ENSEMBLE_SEEDS:
    result = train_single_model_optimized(
        seed, converted_graphs,
        train_influencers_list, val_influencers_list, test_influencers_list,
        all_graphs_data
    )
    ensemble_results.append(result)
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("ENSEMBLE TRAINING COMPLETE")
print("="*60)

## 12. Ensemble Prediction

In [ ]:
print("\nComputing ensemble predictions...")

ensemble_predictions = np.zeros_like(ensemble_results[0]['test_pred'])
for result in ensemble_results:
    ensemble_predictions += result['test_pred']
ensemble_predictions /= len(ensemble_results)

test_true = ensemble_results[0]['test_true']
ensemble_ndcg = compute_ndcg_at_k(test_true, ensemble_predictions, k=50)

print(f"\n" + "="*60)
print("FINAL RESULTS (V3 FIXED: NO TEMPORAL LEAKAGE)")
print("="*60)

print(f"\nIndividual Model Results:")
for result in ensemble_results:
    print(f"  Seed {result['seed']}: Val={result['best_val_ndcg']:.4f}, Test={result['test_ndcg']:.4f}, "
          f"Time={result['avg_epoch_time']:.2f}s/epoch")

avg_test_ndcg = np.mean([r['test_ndcg'] for r in ensemble_results])
std_test_ndcg = np.std([r['test_ndcg'] for r in ensemble_results])
avg_epoch_time = np.mean([r['avg_epoch_time'] for r in ensemble_results])

print(f"\nSingle Model Stats:")
print(f"  Mean NDCG@50: {avg_test_ndcg:.4f} (+/- {std_test_ndcg:.4f})")
print(f"  Avg Epoch Time: {avg_epoch_time:.2f}s")

print(f"\nEnsemble Results:")
print(f"  Ensemble NDCG@50: {ensemble_ndcg:.4f}")

print(f"\nComparison:")
v3_buggy_ndcg = 0.71  # From user's report
print(f"  v3 (BUGGY, temporal leakage): ~{v3_buggy_ndcg:.4f}")
print(f"  v3 FIXED (no leakage): {ensemble_ndcg:.4f}")
print(f"  Difference: {ensemble_ndcg - v3_buggy_ndcg:+.4f}")

print(f"\n" + "="*60)
if ensemble_ndcg >= 0.72:
    print(f"TARGET ACHIEVED! {ensemble_ndcg:.4f} >= 0.72")
elif ensemble_ndcg >= 0.60:
    print(f"HONEST RESULT: {ensemble_ndcg:.4f}")
    print("This is likely the true performance without data leakage.")
elif ensemble_ndcg >= 0.55:
    print(f"BASELINE PERFORMANCE: {ensemble_ndcg:.4f}")
    print("Consider: deeper GCN, attention mechanisms, or more features.")
else:
    print(f"LOW PERFORMANCE: {ensemble_ndcg:.4f}")
    print("Check: Are v3 graphs with progressive temporal features loaded?")
print("="*60)

## 13. Save Results

In [ ]:
import json

results_summary = {
    'version': 'v3_fixed_no_temporal_leakage',
    'ensemble_ndcg': float(ensemble_ndcg),
    'individual_results': [
        {
            'seed': r['seed'],
            'val_ndcg': float(r['best_val_ndcg']),
            'test_ndcg': float(r['test_ndcg']),
            'avg_epoch_time': float(r['avg_epoch_time'])
        }
        for r in ensemble_results
    ],
    'avg_test_ndcg': float(avg_test_ndcg),
    'std_test_ndcg': float(std_test_ndcg),
    'config': {
        'input_dim': INPUT_DIM,
        'gnn_hidden': GNN_HIDDEN,
        'gnn_out': GNN_OUT,
        'rnn_hidden': RNN_HIDDEN,
        'dropout': DROPOUT,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'list_size': LIST_SIZE,
        'leaky_indices_zeroed': LEAKY_INDICES,
        'num_leaky_features': len(LEAKY_INDICES),
        'progressive_temporal_features': True,
        'cached_embeddings_per_epoch': True,
        'graphs_preloaded_to_gpu': True
    },
    'fixes_applied': [
        'Progressive temporal features (no future leakage)',
        'Correct leaky indices (21, 22, 27 added)',
        'GCN embeddings cached per epoch (5-10x speedup)',
        'Graphs pre-loaded to GPU (no transfers)',
        'Verified leakage removal in ALL months'
    ]
}

with open('v3_fixed_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to v3_fixed_results.json")

## 14. Conclusion

In [ ]:
print("\n" + "="*60)
print("SUMMARY - V3 FIXED (NO TEMPORAL LEAKAGE)")
print("="*60)

print("\n1. CRITICAL FIX: PROGRESSIVE TEMPORAL FEATURES")
print("   - OLD BUG: All graphs used temporal from months 0-8")
print("   - NEW: Each graph uses only PAST months")
print("     - Jan graph: No temporal (first month)")
print("     - Feb graph: Temporal from Jan only")
print("     - Sep graph: Temporal from Jan-Aug")
print("   - This eliminates future information leakage")

print("\n2. CORRECTED LEAKY INDICES")
print(f"   - OLD: 14 indices (missing 21, 22, 27)")
print(f"   - NEW: {len(LEAKY_INDICES)} indices (all engagement-related)")
print(f"   - Added: activity_rate (21), posting_consistency (22), log_avg_comments (27)")

print("\n3. PERFORMANCE OPTIMIZATIONS")
print(f"   - GCN embeddings cached per epoch (not per batch)")
print(f"   - Graphs pre-loaded to GPU once")
print(f"   - Expected speedup: 5-10x")
print(f"   - Actual avg epoch time: {avg_epoch_time:.2f}s")

print("\n4. HONEST RESULTS")
print(f"   - Single Model Mean: {avg_test_ndcg:.4f} (+/- {std_test_ndcg:.4f})")
print(f"   - Ensemble NDCG@50: {ensemble_ndcg:.4f}")
print(f"   - Previous v3 (buggy): ~0.71 (INFLATED by leakage)")
print(f"   - Paper target: 0.720")

print("\n5. INTERPRETATION")
if ensemble_ndcg < 0.65:
    print(f"   The lower score ({ensemble_ndcg:.4f}) is expected and honest.")
    print("   The previous 0.71+ was artificially inflated by temporal leakage.")
    print("   ")
    print("   To improve further, consider:")
    print("   - Deeper GCN (3+ layers)")
    print("   - Graph attention networks (GAT)")
    print("   - Multi-head attention for temporal")
    print("   - Feature engineering (more non-leaky features)")
    print("   - Larger training set or data augmentation")

print("\n" + "="*60)